In [5]:
%pip install torch torchaudio torchvision

Note: you may need to restart the kernel to use updated packages.


In [6]:
import json
from pathlib import Path
from typing import List, Dict
from tqdm import tqdm

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [7]:
SA_QUESTION_FILES = {
    "DoS":   Path("DoS_sa_qa/dos_sa_questions.json"),
    "Fuzzy": Path("Fuzzy_sa_qa/fuzzy_sa_questions.json"),
    "Gear":  Path("Gear_sa_qa/gear_sa_questions.json"),
    "RPM":   Path("RPM_sa_qa/rpm_sa_questions.json"),
}

SELECTED_DATASETS = ["DoS", "Fuzzy", "Gear", "RPM"]


In [8]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"  # switch to an open, non-gated model
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
MODEL_TAG = MODEL_ID.split("/")[-1].replace(".", "_").replace("-", "_")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
    pad_token_id=tokenizer.pad_token_id,
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


In [9]:
# # Example: Qwen 3-4B Thinking
# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"  # or "meta-llama/Meta-Llama-3.1-8B-Instruct"
# DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
# MODEL_TAG = MODEL_ID.split("/")[-1].replace(".", "_").replace("-", "_")

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     torch_dtype=DTYPE,
#     device_map="auto",
#     pad_token_id=tokenizer.pad_token_id,
#     trust_remote_code=True,
# )


In [10]:
# MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
# MODEL_TAG = MODEL_ID.split("/")[-1].replace(".", "_").replace("-", "_")

# tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_ID,
#     use_fast=True,
#     trust_remote_code=True,
# )
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     torch_dtype=DTYPE,
#     device_map="auto",
#     pad_token_id=tokenizer.pad_token_id,
#     trust_remote_code=True,
# )


In [11]:
def build_sa_prompt_text(context: str, question: str) -> str:
    """Simple chat prompt for short answers."""
    system_prompt = (
        "You are a CAN bus intrusion-detection analyst. "
        "Given a CAN bus time window and a short-answer question about attack behavior, IDs, timing, payload, or flags, "
        "provide a concise, factual answer."
    )

    user_prompt = (
        "Below is a CAN bus time window. Review the sequence carefully, note anomalies or missing IDs, "
        "and answer the question briefly.\n\n"
        f"{context}\n\n"
        f"Question: {question}\n"
        "Answer concisely in one sentence."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    return tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )


def normalize_text(text: str) -> str:
    return " ".join(text.strip().split()).lower() if text else ""


def query_sa_llm(context: str, question: str, max_new_tokens: int = 64) -> str:
    prompt_text = build_sa_prompt_text(context, question)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()
    return completion


def load_questions(path: Path) -> List[dict]:
    """
    Load questions from .json (list) or .jsonl.
    """
    if not path.exists():
        return []
    if path.suffix == ".json":
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


In [12]:
import torch
print(torch.cuda.is_available())
print(model.device)

False
cpu


In [13]:
for ds_name in SELECTED_DATASETS:
    q_path = SA_QUESTION_FILES[ds_name]
    if not q_path.exists():
        print(f"[WARN] Short-answer questions for {ds_name} not found at {q_path}, skip.")
        continue

    questions = load_questions(q_path)
    print(f"[INFO] {ds_name}: loaded {len(questions)} short-answer questions.")

    out_dir = q_path.parent
    ans_path = out_dir / f"{ds_name.lower()}_sa_answers_{MODEL_TAG}.jsonl"

    ans_path.write_text("", encoding="utf-8")

    with ans_path.open("a", encoding="utf-8") as f:
        for rec in tqdm(questions, desc=f"{ds_name} SA answering"):
            qa_id = rec.get("qa_id")
            metadata = rec.get("metadata", {})
            dataset = metadata.get("dataset", ds_name)
            context = rec["context"]
            question = rec["question"]
            gt = rec.get("answer", "")

            pred = query_sa_llm(context, question)
            answer_rec = {
                "qa_id": qa_id,
                "dataset": dataset,
                "sa_type": rec.get("sa_type"),
                "model": MODEL_ID,
                "llm_answer": pred,
                "ground_truth": gt,
                "is_exact_match": normalize_text(pred) == normalize_text(gt) if gt else False,
            }
            f.write(json.dumps(answer_rec, ensure_ascii=False) + "\n")

    print(f"[INFO] {ds_name}: SA answers saved to {ans_path}")


[INFO] DoS: loaded 183 short-answer questions.


DoS SA answering:   0%|          | 0/183 [00:44<?, ?it/s]


KeyboardInterrupt: 